# GUDS-EDL: Aligned Fair-v2 Experiment Suite & Baselines

This notebook runs the newly updated **`isic_fair_v2_2026_07_09`** protocol. It verifies and runs the experiments incorporating the following core modifications:

1. **Static 2:4 Mask Initialization:** Re-initialized directly with a true static mask from scratch, removing the previous dense pre-training dependency.
2. **RigL Baseline:** Re-implemented using standard magnitude pruning combined with task-gradient regrowth.
3. **Baseline Alignment:** All baseline runs share identical learning rate, loss-scale warmup, and run on the FP32 objective path.
4. **Structural Probe:** Frozen BatchNorm statistics during probing (no BN updates).
5. **Topology Refresh:** Topology is updated/refreshed immediately before launching the prediction evaluation loop.
6. **Dormant Weights:** Excluded from receiving Adam momentum or weight decay to prevent leakage or stale optimization signals.
7. **Saliency Ties & Zero Values:** Correctly handled ties and zero values in first-order gradients.
8. **CUDA Determinism:** Deterministic CUDA execution mode is enabled by default (`MDEP_DETERMINISTIC=1`).
9. **OOD/PAD Checkpoint Safety:** Checkpoints from older runs are rejected by default to avoid mixing legacy/provisional results.
10. **Manuscript Alignment:** Updates corresponding to evidence cap, 19 sparse convolutions, data augmentation, and fairness protocol.
11. **Provisional Marking:** Marks all legacy metrics as provisional, presenting only `fair-v2` confirmed results as verified conclusions.

## Step 1: Clone Repository & Setup Directories

Clone the repository and set up the active directory.

In [ ]:
# 1. Clone repository
!git clone https://github.com/minhduc110207/MDEP-Microglial-Driven-Evidential-Pruning.git
%cd MDEP-Microglial-Driven-Evidential-Pruning

## Step 2: Dataset Verification and Path Mapping

Verify the presence of both required datasets:
- ISIC 2024 challenge dataset: `/kaggle/input/competitions/isic-2024-challenge`
- Skin Cancer (PAD-UFES) dataset: `/kaggle/input/datasets/mahdavi1202/skin-cancer`

If the datasets are added but located at alternative Kaggle folders (e.g., direct root), this cell automatically creates symlinks to map them to the expected paths.

In [ ]:
import os
from pathlib import Path

print("=== Checking and Mapping Kaggle Datasets ===")

# 1. Check & map ISIC 2024 dataset
isic_target = Path("/kaggle/input/competitions/isic-2024-challenge")
isic_sources = [
    Path("/kaggle/input/isic-2024-challenge"),
    Path("/kaggle/input/competitions/isic-2024-challenge")
]
if not isic_target.exists():
    linked = False
    for src in isic_sources:
        if src.exists() and src != isic_target:
            print(f"[INFO] Found ISIC 2024 at {src}. Creating symlink at {isic_target}...")
            os.makedirs(isic_target.parent, exist_ok=True)
            try:
                os.symlink(src, isic_target)
                linked = True
                break
            except Exception as e:
                print(f"[WARN] Failed to create symlink: {e}")
    if not linked:
        print("[WARNING] ISIC 2024 dataset not found. Please attach the 'isic-2024-challenge' competition to this Notebook.")
else:
    print("[OK] ISIC 2024 challenge dataset is correctly mounted.")

# 2. Check & map Skin Cancer (PAD-UFES) dataset
pad_target = Path("/kaggle/input/datasets/mahdavi1202/skin-cancer")
pad_sources = [
    Path("/kaggle/input/skin-cancer"),
    Path("/kaggle/input/skin-cancer-dataset"),
    Path("/kaggle/input/datasets/mahdavi1202/skin-cancer")
]
if not pad_target.exists():
    linked = False
    for src in pad_sources:
        if src.exists() and src != pad_target:
            print(f"[INFO] Found Skin Cancer (PAD-UFES) at {src}. Creating symlink at {pad_target}...")
            os.makedirs(pad_target.parent, exist_ok=True)
            try:
                os.symlink(src, pad_target)
                linked = True
                break
            except Exception as e:
                print(f"[WARN] Failed to create symlink: {e}")
    if not linked:
        print("[WARNING] Skin Cancer (PAD-UFES) dataset not found. Please attach the 'mahdavi1202/skin-cancer' dataset to this Notebook.")
else:
    print("[OK] Skin Cancer (PAD-UFES) dataset is correctly mounted.")

print("=== Dataset check complete ===")

## Step 3: Smoke Test

Run a quick 1-epoch dry-run to verify environment sanity.

In [ ]:
# 3. Quick smoke test
!python experiments/run_kaggle_paper_suite.py --smoke

## Step 4: Train Modified ISIC 2024 Baseline & Proposed Models

Train the baseline and proposed models under the aligned `isic_fair_v2_2026_07_09` protocol using 3 seeds: 42, 123, and 456. We use `--suite all` to run all modified baselines, ablations, and the proposed model.

In [ ]:
# 4. Train all baselines & proposed models (seeds 42, 123, 456) under the fair-v2 protocol
!MDEP_DETERMINISTIC=1 python -u experiments/isic_paper_experiments.py \
  --suite all \
  --epochs 40 \
  --batch_size 32 \
  --lr 4e-5 \
  --seeds 42 123 456 \
  --split_seed 42 \
  --subsample_scope train \
  --subsample_ratio 20 \
  --structural_proxy_batches 4 \
  --checkpoint_selection last \
  --run_suffix _fair_v2

## Step 5: Post-hoc OOD / External Validation

Evaluate out-of-distribution performance on the external PAD-UFES dataset. We load checkpoints directly from the newly trained fair-v2 paths.

In [ ]:
# 5. Post-hoc OOD evaluation for each seed
for seed in [42, 123, 456]:
    print(f"\nEvaluating Out-Of-Distribution (OOD) for Seed {seed}...")
    !python -u experiments/run_external_validation.py \
      --model_path "/kaggle/working/paper_experiment_outputs/isic/full_guds_fair_v2/seed_{seed}/model_state.pth" \
      --seed {seed} \
      --split_seed 42 \
      --custom_image_folder /kaggle/input/datasets/mahdavi1202/skin-cancer \
      --pad_ufes_csv /kaggle/input/datasets/mahdavi1202/skin-cancer/metadata.csv \
      --pad_ufes_partition imgs_part_3 \
      --knn_primary_layer layer3 \
      --primary_ood_score knn_layer3

## Step 6: Leakage-Safe PAD Adaptation

Adapt the models to PAD-UFES using frozen backbones and a multi-fold outer/inner calibration scheme.

In [ ]:
# 6. Run the leakage-safe PAD adaptation protocol
!python experiments/run_pad_adaptation.py \
  --pad_root /kaggle/input/datasets/mahdavi1202/skin-cancer \
  --pad_csv /kaggle/input/datasets/mahdavi1202/skin-cancer/metadata.csv \
  --partition all \
  --model_path '/kaggle/working/paper_experiment_outputs/isic/full_guds_fair_v2/seed_{seed}/model_state.pth' \
  --seeds 42 123 456 \
  --target_mode diagnosis6 \
  --feature_layer auto \
  --head linear \
  --outer_folds 5 \
  --inner_folds 3 \
  --fairness_min_group_size 20 \
  --fairness_min_class_size 10 \
  --target_sensitivity 0.80 \
  --bootstrap_repeats 1000 \
  --train_domain_head

## Step 7: Constrained Layer-4 KD Adaptation

Fine-tune only layer-4 weights under knowledge distillation constraint as a secondary stage.

In [ ]:
# 7. Run the constrained layer 4 KD stage
!python experiments/run_pad_layer4_kd.py \
  --pad_root /kaggle/input/datasets/mahdavi1202/skin-cancer \
  --pad_csv /kaggle/input/datasets/mahdavi1202/skin-cancer/metadata.csv \
  --partition all \
  --model_path '/kaggle/working/paper_experiment_outputs/isic/full_guds_fair_v2/seed_{seed}/model_state.pth' \
  --seeds 42 123 456 \
  --target_mode diagnosis6 \
  --outer_folds 5 \
  --inner_folds 3 \
  --epochs 12 \
  --lr 1e-5 \
  --kd_weight 2.0 \
  --kd_temperature 2.0

## Step 8: Summarize and View Metric Results

Aggregate results across seeds to display the paper-facing tables.

In [ ]:
# 8. Summarize all metrics
!python experiments/summarize_results.py